# Maamoura controlled factorial: Phase 1 parent x Phase 2 Huber delta

This notebook completes the minimal controlled comparison needed to separate
the effect of the Phase 1 parent checkpoint from the Phase 2 Huber threshold.
Phase 1 is held fixed to the original natural-sampling run with `delta=1` and
seed 42. The two factors are:

- Phase 1 parent: `best_any.ckpt` or `best_slope.ckpt` (both selected on VAL);
- Phase 2 Huber delta: 1 or 3.

The `best_any x delta=3` production run and the `best_slope x delta=3` run are
reused after exact lineage checks. Only the two missing delta=1 Phase 2 runs
are trained. All Phase 2 checkpoints are selected on VAL using
`best_compromise.ckpt`; TEST is opened only after the four-cell registry is
complete and frozen.

The earlier `Phase 1 delta=3` experiment is not part of this 2x2 comparison:
it changes the Phase 1 objective and is retained as a separate control.


## 1. Imports, paths and immutable experiment design

Restarting is safe: a completed run is reused only when its parent SHA-256,
seed, disturbance settings and Huber delta match the prescribed cell.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gc, hashlib, importlib.util, json, sys

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
EXPERIMENT = PROJECT / "Ablations" / "Maamoura_Phase2_Delta_Parent_Factorial_Seed42"
RUN_ROOT = EXPERIMENT / "runs"
PREDICTION_ROOT = EXPERIMENT / "test_predictions"
OUTPUT_ROOT = EXPERIMENT / "article_plots"
for path in (RUN_ROOT, PREDICTION_ROOT, OUTPUT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

BASE_RUNNER = PROJECT / "Ablations" / "Phase2_Hypothesis_Validation" / "runtime" / "run_phase2_hypothesis_ablation.py"
ENGINE_PATH = PROJECT / "Source" / "Project" / "low_canopy_growthloss_ablation_runner.py"
ADAPTER_PATH = PROJECT / "Source" / "Project" / "aoi_masked_phase2_adapter.py"
CONFIG_PATH = PROJECT / "Source" / "Project" / "b4_c15_config.py"

PHASE1_RUN = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Ablations\Phase1_Sampler_Cursor_Ablation\runs\maamoura\final_catalog\MAAMOURA_PHASE1_NATURAL_SAMPLING_V2_SEED42")
P1_ANY = PHASE1_RUN / "checkpoints" / "best_any.ckpt"
P1_SLOPE = PHASE1_RUN / "checkpoints" / "best_slope.ckpt"

PRODUCTION_P2 = PROJECT / "Ablations" / "Phase2_D_K_Lambda_Delta_VAL_Only" / "runs" / "maamoura" / "MAAMOURA_B4_C15_CTRL_GL000_D2_K2_HD3_SEED42"
SLOPE_DELTA3_P2 = PROJECT / "Ablations" / "Uniform_Checkpoint_Comparison_Seed42" / "runs" / "maamoura" / "training" / "MAAMOURA_B4_C15_UNIFORM_P1_BEST_SLOPE_P2_BEST_COMPROMISE_SEED42_SEED42"

SEED = 42
EXPECTED_TEST_N = 1799
RUN_TRAINING = True
AUTHORIZE_TEST_EVALUATION = True

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

base = load_module("maamoura_factorial_base", BASE_RUNNER)
cfgmod = load_module("maamoura_factorial_config", CONFIG_PATH)
print("Experiment:", EXPERIMENT)


## 2. Preflight and four prespecified cells

This is a clean 2x2 comparison because the Phase 1 training run (`delta=1`),
data, split, architecture and seed are identical in all four cells.


In [ ]:
for required in (P1_ANY, P1_SLOPE, BASE_RUNNER, ENGINE_PATH, ADAPTER_PATH, CONFIG_PATH):
    if not required.is_file():
        raise FileNotFoundError(required)

p1_config_path = PHASE1_RUN / "artifacts" / "run_config.json"
if not p1_config_path.is_file():
    raise FileNotFoundError(p1_config_path)
p1_args = json.loads(p1_config_path.read_text(encoding="utf-8"))["args"]
p1_checks = {
    "seed_42": int(p1_args["seed"]) == 42,
    "natural_sampling": p1_args["train_sampler_mode"] == "natural",
    "phase1_huber_delta_1": float(p1_args["huber_beta"]) == 1.0,
    "test_disabled": not bool(p1_args["eval_test_at_end"]),
}
if not all(p1_checks.values()):
    raise RuntimeError(f"Phase 1 lineage mismatch: {p1_checks}")

VARIANTS = {
    "best_any__p2_delta3": {"parent_rule": "best_any", "parent": P1_ANY, "p2_delta": 3.0, "run_dir": PRODUCTION_P2, "reuse": True},
    "best_slope__p2_delta3": {"parent_rule": "best_slope", "parent": P1_SLOPE, "p2_delta": 3.0, "run_dir": SLOPE_DELTA3_P2, "reuse": True},
    "best_any__p2_delta1": {"parent_rule": "best_any", "parent": P1_ANY, "p2_delta": 1.0, "run_dir": RUN_ROOT / "best_any__p2_delta1", "reuse": False},
    "best_slope__p2_delta1": {"parent_rule": "best_slope", "parent": P1_SLOPE, "p2_delta": 1.0, "run_dir": RUN_ROOT / "best_slope__p2_delta1", "reuse": False},
}

for name, variant in VARIANTS.items():
    variant["parent_sha256"] = file_sha256(variant["parent"])
    variant["checkpoint"] = variant["run_dir"] / "checkpoints" / "best_compromise.ckpt"

design = pd.DataFrame([
    {"variant": name, "phase1_delta": 1.0, "phase1_parent": v["parent_rule"],
     "phase1_sha256": v["parent_sha256"], "phase2_delta": v["p2_delta"],
     "existing_checkpoint": v["checkpoint"].is_file(), "run_dir": str(v["run_dir"])}
    for name, v in VARIANTS.items()
])
design.to_csv(EXPERIMENT / "00_prespecified_factorial_design.csv", index=False)
display(design)
print("Phase 1 checks:", p1_checks)


## 3. Validate reusable runs and train only the two missing delta=1 runs

No Phase 1 training is performed. The production and existing
`best_slope x delta=3` checkpoints are never overwritten.


In [ ]:
def prepare_variant(name, variant):
    engine = load_module(f"factorial_engine_{name}", ENGINE_PATH)
    adapter = load_module(f"factorial_adapter_{name}", ADAPTER_PATH)
    cfg = engine.FORESTS["maamoura"]
    cfg["parent"] = variant["parent"]
    cfg["parent_sha"] = variant["parent_sha256"]
    cfg["ablation_family"] = "Maamoura_Phase2_Delta_Parent_Factorial_Seed42"
    candidate_spec = {"drop_m": 2.0, "K": 2, "lambda_growth": 0.0, "huber_delta": variant["p2_delta"]}
    engine.LOW_CANOPY_CANDIDATES = {name: candidate_spec}
    engine.roots = lambda _forest: (variant["run_dir"].parent, EXPERIMENT / "reports" / name)
    modules = adapter.prepare_aoi_masked_modules(engine)
    return engine, modules, cfg, candidate_spec

def validate_existing(name, variant):
    checkpoint = variant["checkpoint"]
    config_path = variant["run_dir"] / "config.json"
    if not checkpoint.is_file() or not config_path.is_file():
        return False
    saved = json.loads(config_path.read_text(encoding="utf-8"))
    growth = saved["growth_loss_provenance"]
    checks = {
        "parent_sha": saved["phase1_parent"]["source_sha256"] == variant["parent_sha256"],
        "seed": int(saved["seed"]) == SEED,
        "delta": float(saved["huber_delta"]) == variant["p2_delta"],
        "D": float(growth["persistent_drop_m"]) == 2.0,
        "K": int(growth["persistent_required_consecutive_flags"]) == 2,
        "lambda_temp": float(saved["lambda_growth"]) == 0.0,
    }
    if not all(checks.values()):
        raise RuntimeError(f"{name}: incompatible existing run: {checks}")
    print("REUSE", name, checkpoint, flush=True)
    return True

def train_variant(name, variant):
    engine, modules, cfg, spec = prepare_variant(name, variant)
    shots, records = base.harmonize(engine, modules, "maamoura")
    train_ds = modules["B4SequenceCropDataset"](
        records["train"], shots, crop_size=96, samples_per_epoch=264,
        drop_channels=(), seed=SEED, center_on_gedi=True,
        balanced_height_anchors=True, height_bins=cfg["height_bins"],
    )
    val_ds = modules["B4SequenceCropDataset"](
        records["val"], shots, crop_size=96,
        samples_per_epoch=max(132, 4 * len(records["val"])),
        drop_channels=(), seed=SEED + 10000, center_on_gedi=True,
        balanced_height_anchors=False, height_bins=cfg["height_bins"],
    )
    model = engine.fresh_model(cfg, modules)
    modules["train"](
        model=model, train_dataset=train_ds, val_dataset=val_ds,
        official_repo=engine.OFFICIAL_REPO, phase1_checkpoint=cfg["parent"],
        run_dir=variant["run_dir"], device=engine.DEVICE, seed=SEED,
        batch_size=1, max_steps=cfg["max_steps"], val_every_steps=66,
        patience_evals=20, learning_rate=1e-4, weight_decay=5e-3,
        lambda_growth=0.0, lambda_slope=0, lambda_std=0, lambda_bias=0,
        lambda_anti_zero=0, supervised_loss_name="huber",
        huber_delta=variant["p2_delta"], height_weight_mode="none",
        slope_min=0, slope_max=2, disturbance_indicator=-1,
        disturbance_rule="persistent_running_max", persistent_drop_m=2.0,
        persistent_required_consecutive_flags=2, full_disturbance_window=True,
        min_height=cfg["eval_min"], max_height=cfg["eval_max"],
        checkpoint_min_slope=0, checkpoint_min_std_ratio=0,
        checkpoint_max_std_ratio=2, checkpoint_max_abs_bias=5,
        warmup_cycles=3, plateau_patience=8, plateau_factor=0.5,
        lr_min=1e-6, grad_clip=1, allow_resume=True, reuse_completed=True,
    )
    del model, train_ds, val_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

for name, variant in VARIANTS.items():
    complete = validate_existing(name, variant)
    if not complete:
        if variant["reuse"]:
            raise FileNotFoundError(f"Required reusable run missing: {variant['run_dir']}")
        if not RUN_TRAINING:
            raise RuntimeError(f"Training disabled but {name} is missing")
        print("TRAIN", name, flush=True)
        train_variant(name, variant)
        if not validate_existing(name, variant):
            raise RuntimeError(f"{name}: training ended without a valid best_compromise checkpoint")


## 4. Freeze the four VAL-selected checkpoints

The registry must be complete before any TEST evaluation begins.


In [ ]:
registry_rows = []
for name, variant in VARIANTS.items():
    checkpoint = variant["checkpoint"]
    state = torch.load(checkpoint, map_location="cpu", weights_only=False)
    metrics = state.get("metrics", {})
    registry_rows.append({
        "variant": name, "phase1_delta": 1.0,
        "phase1_parent_rule": variant["parent_rule"],
        "phase1_checkpoint": str(variant["parent"]),
        "phase1_sha256": variant["parent_sha256"],
        "phase2_delta": variant["p2_delta"],
        "phase2_checkpoint_rule": "best_compromise.ckpt selected on VAL only",
        "phase2_checkpoint": str(checkpoint), "phase2_sha256": file_sha256(checkpoint),
        "selection_split": "VAL", "test_used_for_selection": False,
        **{f"val_{key}": metrics.get(key) for key in ("n", "mae", "rmse", "r2", "bias", "slope", "std_ratio")},
    })
registry = pd.DataFrame(registry_rows)
registry.to_csv(EXPERIMENT / "01_frozen_checkpoint_registry.csv", index=False)
display(registry)
if len(registry) != 4 or not registry["phase2_checkpoint"].map(lambda p: Path(p).is_file()).all():
    raise RuntimeError("TEST blocked: the four-cell registry is incomplete")
print("[PASS] Four VAL-selected checkpoints frozen before TEST.")


## 5. Evaluate the four cells on the identical frozen TEST support

These TEST results are a diagnostic of the prespecified experiment. They must
not be used to iteratively tune further variants.


In [ ]:
def support_fingerprint(ids):
    return hashlib.sha256("\n".join(sorted(map(str, ids))).encode("utf-8")).hexdigest()

def evaluate_variant(row):
    name = row.variant
    variant = VARIANTS[name]
    engine, modules, cfg, _ = prepare_variant(name, variant)
    checkpoint = Path(row.phase2_checkpoint)
    if file_sha256(checkpoint) != row.phase2_sha256:
        raise RuntimeError(f"{name}: checkpoint hash changed")
    state = torch.load(checkpoint, map_location="cpu", weights_only=False)
    _, shots, records = engine.build_data("maamoura", modules, include_test=True)
    model = engine.fresh_model(cfg, modules)
    model.prediction_head.load_state_dict(state["prediction_head"], strict=True)
    model.eval()
    _, nearest = modules["evaluate_full_patch_temporal_nearest"](
        model=model, records=records["test"], shots=shots, device=engine.DEVICE,
        split="test", drop_channels=(), min_height=cfg["eval_min"],
        max_height=cfg["eval_max"], progress_every=1,
    )
    nearest["aux_shot_uid"] = nearest["aux_shot_uid"].astype(str)
    if len(nearest) != EXPECTED_TEST_N or not nearest["aux_shot_uid"].is_unique:
        raise RuntimeError(f"{name}: unexpected TEST support")
    nearest["split"] = "test"
    nearest["variant"] = name
    nearest["checkpoint_sha256"] = row.phase2_sha256
    out_dir = PREDICTION_ROOT / name
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / "test_unique_nearest.csv.gz"
    nearest.to_csv(out, index=False, compression="gzip")
    lineage = {
        "variant": name, "n": len(nearest),
        "support_sha256": support_fingerprint(nearest["aux_shot_uid"]),
        "phase1_delta": 1.0, "phase1_parent_rule": variant["parent_rule"],
        "phase1_parent_sha256": variant["parent_sha256"],
        "phase2_delta": variant["p2_delta"], "phase2_rule": "best_compromise.ckpt",
        "phase2_sha256": row.phase2_sha256, "selection_split": "VAL",
        "evaluation_split": "TEST",
    }
    (out_dir / "lineage.json").write_text(json.dumps(lineage, indent=2), encoding="utf-8")
    del model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return out

if AUTHORIZE_TEST_EVALUATION:
    prediction_paths = [evaluate_variant(row) for row in registry.itertuples(index=False)]
    supports = [json.loads((p.parent / "lineage.json").read_text(encoding="utf-8"))["support_sha256"] for p in prediction_paths]
    if len(set(supports)) != 1:
        raise RuntimeError("The four variants do not use the identical TEST support")
    print("[PASS] Identical TEST support across all four variants:", supports[0])


## 6. Generate the same six diagnostic graph families for every cell

Each variant receives scatter, height-distribution, height-class error,
signed-error, residual and absolute-error-CDF outputs in PNG, SVG and PDF.


In [ ]:
SITES = {
    name: {
        "ecosystem": "Maamoura_factorial",
        "input": PREDICTION_ROOT / name / "test_unique_nearest.csv.gz",
        "eval_min": 2.0, "eval_max": 20.0,
        "height_bins": np.asarray([0, 5, 10, 15, 20], float),
        "expected_n": EXPECTED_TEST_N,
    }
    for name in VARIANTS
}


In [ ]:
# 2 — Shared loading, metrics, plotting and export functions
GEDI_HIST = "lightsteelblue"
GEDI_BAR = "steelblue"
PRED_COLOR = "darkorange"
MEDIAN_COLOR = "red"
ACCENT_BLUE = "#1f77b4"
ACCENT_ORANGE = "#ff7f0e"

# Fixed publication geometry shared by panels (a) and (b).  The main
# plotting rectangle has exactly the same left/right coordinates in both
# PDFs; the scatter colorbar occupies a reserved strip outside it.
FIGURE_WIDTH_IN = 7.50
MAIN_AX_LEFT = 0.145
MAIN_AX_WIDTH = 0.690
TOP_AX_RECT = [MAIN_AX_LEFT, 0.245, MAIN_AX_WIDTH, 0.665]
SCATTER_AX_RECT = [MAIN_AX_LEFT, 0.145, MAIN_AX_WIDTH, 0.790]
COLORBAR_AX_RECT = [0.865, 0.145, 0.028, 0.790]

# Publication typography: axis labels and tick numerals use exactly the same
# font family, weight and size on primary, secondary and colorbar axes.
AXIS_FONT_FAMILY = "DejaVu Sans"
AXIS_TEXT_SIZE = 11
plt.rcParams.update({
    "font.family": AXIS_FONT_FAMILY,
    "font.size": AXIS_TEXT_SIZE,
    "axes.labelsize": AXIS_TEXT_SIZE,
    "axes.labelweight": "normal",
    "xtick.labelsize": AXIS_TEXT_SIZE,
    "ytick.labelsize": AXIS_TEXT_SIZE,
})


def harmonize_axis_typography(*axes):
    """Apply one publication font to axis labels and all tick numerals."""
    for axis in axes:
        if axis is None:
            continue
        axis.xaxis.label.set_fontfamily(AXIS_FONT_FAMILY)
        axis.yaxis.label.set_fontfamily(AXIS_FONT_FAMILY)
        axis.xaxis.label.set_fontsize(AXIS_TEXT_SIZE)
        axis.yaxis.label.set_fontsize(AXIS_TEXT_SIZE)
        axis.tick_params(axis="both", which="both", labelsize=AXIS_TEXT_SIZE)
        for label in axis.get_xticklabels() + axis.get_yticklabels():
            label.set_fontfamily(AXIS_FONT_FAMILY)
            label.set_fontweight("normal")


def export_figure(fig, output_dir: Path, stem: str):
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for suffix in ("png", "svg", "pdf"):
        path = output_dir / f"{stem}.{suffix}"
        fig.savefig(
            path,
            dpi=1200 if suffix == "png" else None,
            bbox_inches=None,
            pad_inches=0.0,
            facecolor="white",
        )
        paths.append(path)
    print("Saved:", ", ".join(str(path) for path in paths))
    return paths


def load_phase2_test(forest: str) -> pd.DataFrame:
    cfg = SITES[forest]
    frame = pd.read_csv(cfg["input"])
    if not frame["split"].astype(str).str.lower().eq("test").all():
        raise AssertionError(f"{forest}: non-TEST rows found")
    if frame["aux_shot_uid"].astype(str).duplicated().any():
        raise AssertionError(f"{forest}: duplicated aux_shot_uid after unique-nearest")
    frame["rh95"] = pd.to_numeric(frame["rh95"], errors="coerce")
    frame["prediction"] = pd.to_numeric(
        frame["pred_on_growthloss"], errors="coerce"
    )
    valid = (
        np.isfinite(frame["rh95"])
        & np.isfinite(frame["prediction"])
        & frame["rh95"].between(cfg["eval_min"], cfg["eval_max"], inclusive="both")
    )
    frame = frame.loc[valid].copy().reset_index(drop=True)
    if frame.empty:
        raise RuntimeError(f"{forest}: no valid Phase 2 TEST observation")
    if len(frame) != cfg["expected_n"]:
        raise RuntimeError(f"{forest}: valid TEST n={len(frame)} != expected {cfg['expected_n']}")
    return frame


def compute_metrics(frame: pd.DataFrame) -> dict:
    true = frame["rh95"].to_numpy(float)
    pred = frame["prediction"].to_numpy(float)
    error = pred - true
    sst = float(np.sum((true - true.mean()) ** 2))
    true_std = float(np.std(true, ddof=0))
    pred_std = float(np.std(pred, ddof=0))
    corr = (
        float(np.corrcoef(true, pred)[0, 1])
        if len(true) > 1 and true_std > 0 and pred_std > 0 else np.nan
    )
    return {
        "n": int(len(frame)),
        "mae": float(np.mean(np.abs(error))),
        "rmse": float(np.sqrt(np.mean(error ** 2))),
        "r2": float(1.0 - np.sum(error ** 2) / sst) if sst > 0 else np.nan,
        "bias": float(np.mean(error)),
        "slope": (
            float(np.polyfit(true, pred, 1)[0])
            if len(true) > 1 and true_std > 0 else np.nan
        ),
        "corr": corr,
        "std_ratio": pred_std / true_std if true_std > 0 else np.nan,
    }


def metric_text(metric: dict) -> str:
    return (
        f"n={metric['n']:,}\n"
        f"MAE={metric['mae']:.2f} m\n"
        f"RMSE={metric['rmse']:.2f} m\n"
        f"R²={metric['r2']:.2f}\n"
        f"Bias={metric['bias']:+.2f} m\n"
        f"Slope={metric['slope']:.2f}\n"
        f"Corr={metric['corr']:.2f}\n"
        f"Std ratio={metric['std_ratio']:.2f}"
    )


def error_ylim(error):
    finite = np.asarray(error, dtype=float)
    finite = finite[np.isfinite(finite)]
    low, high = np.quantile(finite, [0.003, 0.997])
    return max(min(low * 1.30, -8.0), -45.0), min(max(high * 1.15, 6.0), 25.0)


def density_per_1m_cell(true, pred, axis_max):
    edges = np.arange(0.0, axis_max + 1.0001, 1.0)
    counts, _, _ = np.histogram2d(true, pred, bins=(edges, edges))
    ix = np.clip(np.searchsorted(edges, true, side="right") - 1, 0, len(edges) - 2)
    iy = np.clip(np.searchsorted(edges, pred, side="right") - 1, 0, len(edges) - 2)
    return counts[ix, iy]


def nice_scale(max_value, target_intervals=5):
    if not np.isfinite(max_value) or max_value <= 0:
        return 1.0, np.asarray([0, 1])
    raw = max_value / target_intervals
    magnitude = 10.0 ** np.floor(np.log10(raw))
    fraction = raw / magnitude
    nice = 1.0 if fraction <= 1 else 2.0 if fraction <= 2 else 5.0 if fraction <= 5 else 10.0
    step = max(1.0, nice * magnitude)
    upper = float(np.ceil(max_value / step) * step)
    return upper, np.arange(0.0, upper + 0.5 * step, step)


def plot_scatter(frame, forest, output_dir):
    cfg = SITES[forest]
    axis_max = cfg["eval_max"]
    true = frame["rh95"].to_numpy(float)
    pred = frame["prediction"].to_numpy(float)
    # Canopy height is physically non-negative. Clip only for graphical
    # display; compute_metrics(frame) below continues to use raw predictions.
    pred_plot = np.clip(pred, 0.0, axis_max)
    density = density_per_1m_cell(true, pred_plot, axis_max)
    order = np.argsort(density, kind="mergesort")
    upper, ticks = nice_scale(float(density.max()))
    metric = compute_metrics(frame)

    # Keep the full horizontal extent used by panel (a), while making panel
    # (b) only moderately shorter.  The axes are regenerated at this aspect
    # ratio; LaTeX must not impose a second, distorting height constraint.
    fig = plt.figure(figsize=(FIGURE_WIDTH_IN, 5.65))
    ax = fig.add_axes(SCATTER_AX_RECT)
    points = ax.scatter(
        true[order], pred_plot[order], c=density[order],
        cmap="viridis", norm=Normalize(0.0, upper),
        s=float(np.clip(45000.0 / len(frame), 7.0, 16.0)),
        alpha=0.90, edgecolors="none", rasterized=True,
    )
    ax.plot([0, axis_max], [0, axis_max], "k--", linewidth=1.0)
    ax.set(xlim=(0, axis_max), ylim=(0, axis_max))
    # Rectangular plotting area preserves readable width in the article.
    ax.set_xlabel("RH95 from GEDI waveforms (m)")
    ax.set_ylabel("Predicted height (m)")
    ax.grid(True, color="0.88", linewidth=0.55)
    ax.text(
        0.025, 0.975, metric_text(metric),
        transform=ax.transAxes, ha="left", va="top", fontsize=9,
        bbox={
            "facecolor": "white", "edgecolor": "0.72",
            "alpha": 0.70, "pad": 2.2,
        },
    )
    cax = fig.add_axes(COLORBAR_AX_RECT)
    colorbar = fig.colorbar(points, cax=cax)
    colorbar.set_ticks(ticks)
    colorbar.set_ticklabels([f"{int(value):,}" for value in ticks])
    colorbar.ax.minorticks_off()
    harmonize_axis_typography(ax, colorbar.ax)
    export_figure(fig, output_dir, "01_scatter_observed_predicted")
    plt.show()
    plt.close(fig)


def class_errors(frame, edges):
    true = frame["rh95"].to_numpy(float)
    error = frame["prediction"].to_numpy(float) - true
    data, positions, labels, counts = [], [], [], []
    for index, (low, high) in enumerate(zip(edges[:-1], edges[1:])):
        mask = (true >= low) & (true < high if index < len(edges) - 2 else true <= high)
        counts.append(int(mask.sum()))
        labels.append(f"{low:g}–{high:g}")
        if mask.any():
            data.append(error[mask])
            positions.append((low + high) / 2.0)
    return error, data, positions, labels, counts


def plot_height_distribution_error(frame, forest, output_dir):
    cfg = SITES[forest]
    true = frame["rh95"].to_numpy(float)
    pred = frame["prediction"].to_numpy(float)
    # Display classes use the conventional 0--5 m first label. The loaded
    # evaluation frame is already filtered to the frozen >=2 m domain, so
    # this relabelling does not add observations or change any metric.
    error, boxes, positions, _, _ = class_errors(frame, cfg["height_bins"])
    # Canopy height is physically non-negative. Clip only for graphical
    # display; the reported metrics continue to use raw predictions.
    pred_display = np.clip(pred, 0.0, cfg["eval_max"])
    hist_min = cfg["eval_min"]
    hist_edges = np.arange(hist_min, cfg["eval_max"] + 1.0001, 1.0)

    # Martin-style compact upper panel: same source width as panel (b),
    # but substantially shorter vertically.
    fig = plt.figure(figsize=(FIGURE_WIDTH_IN, 3.05))
    count_ax = fig.add_axes(TOP_AX_RECT)
    count_ax.hist(true, bins=hist_edges, color=GEDI_HIST, alpha=0.92, label="GEDI Height")
    count_ax.hist(
        pred_display, bins=hist_edges,
        histtype="step", color=PRED_COLOR, linewidth=1.8, label="Predicted Height",
    )
    error_ax = count_ax.twinx()
    if boxes:
        error_ax.boxplot(
            boxes, positions=positions, widths=0.65, patch_artist=True,
            showfliers=False, manage_ticks=False,
            medianprops={"color": MEDIAN_COLOR, "linewidth": 1.6},
            boxprops={"facecolor": "black", "edgecolor": "black"},
            whiskerprops={"color": "black"}, capprops={"color": "black"},
        )
    error_ax.axhline(0, color="0.35", linestyle=":", linewidth=1.3)
    error_limits = error_ylim(error)
    error_ax.set_ylim(*error_limits)
    zero_fraction = (0.0 - error_limits[0]) / (error_limits[1] - error_limits[0])
    count_ax.set_ylabel("Count")
    error_ax.set_ylabel("Error (m)")
    # Keep the vertical title clear of the largest count tick labels.
    count_ax.yaxis.set_label_coords(-0.105, zero_fraction)
    error_ax.yaxis.set_label_coords(1.075, zero_fraction)
    # Display the physical CHM axis and conventional 5 m graduations from 0.
    count_ax.set_xlim(0.0, cfg["eval_max"])
    x_ticks = list(cfg["height_bins"])
    count_ax.set_xticks([x for x in x_ticks if 0.0 <= x <= cfg["eval_max"]])
    count_ax.set_xlabel("")  # Keep numeric ticks; omit redundant panel-(a) axis title.
    count_ax.legend(loc="upper right", framealpha=0.70)
    harmonize_axis_typography(count_ax, error_ax)
    export_figure(fig, output_dir, "02_height_distribution_and_error")
    plt.show()
    plt.close(fig)


def plot_height_error_bins(frame, forest, output_dir):
    cfg = SITES[forest]
    error, boxes, positions, labels, counts = class_errors(frame, cfg["height_bins"])
    centers = 0.5 * (cfg["height_bins"][:-1] + cfg["height_bins"][1:])
    fig, count_ax = plt.subplots(figsize=(10.5, 5.2))
    count_ax.bar(centers, counts, width=4.6, color=GEDI_BAR, alpha=0.82)
    error_ax = count_ax.twinx()
    if boxes:
        error_ax.boxplot(
            boxes, positions=positions, widths=1.85, patch_artist=True,
            showfliers=False, manage_ticks=False,
            medianprops={"color": MEDIAN_COLOR, "linewidth": 1.6},
            boxprops={"facecolor": "white", "edgecolor": "0.35"},
            whiskerprops={"color": "0.35"}, capprops={"color": "0.35"},
        )
    error_ax.axhline(0, color=ACCENT_BLUE, linestyle="--", linewidth=1.1)
    error_ax.set_ylim(*error_ylim(error))
    count_ax.set_xlim(0, cfg["eval_max"])
    count_ax.set_xticks(centers, labels)
    count_ax.set_xlabel("")
    export_figure(fig, output_dir, "03_height_class_errors")
    plt.show()
    plt.close(fig)


def plot_signed_error(frame, forest, output_dir):
    error = frame["prediction"].to_numpy(float) - frame["rh95"].to_numpy(float)
    fig, ax = plt.subplots(figsize=(7.8, 4.5))
    ax.hist(error, bins=45, alpha=0.85, color=GEDI_BAR, edgecolor="white")
    ax.axvline(0, color="black", linestyle="--", linewidth=1.1)
    ax.axvline(np.median(error), color=MEDIAN_COLOR, linewidth=1.7,
               label=f"Median={np.median(error):+.2f} m")
    ax.axvline(np.mean(error), color=ACCENT_ORANGE, linestyle="--", linewidth=1.4,
               label=f"Mean={np.mean(error):+.2f} m")
    ax.set_xlabel("Prediction − GEDI (m)")
    ax.legend(framealpha=0.70)
    export_figure(fig, output_dir, "04_signed_error_distribution")
    plt.show()
    plt.close(fig)


def plot_residuals(frame, forest, output_dir):
    cfg = SITES[forest]
    true = frame["rh95"].to_numpy(float)
    error, _, _, _, _ = class_errors(frame, cfg["height_bins"])
    centers, medians = [], []
    for index, (low, high) in enumerate(zip(cfg["height_bins"][:-1], cfg["height_bins"][1:])):
        mask = (true >= low) & (true < high if index < len(cfg["height_bins"]) - 2 else true <= high)
        if mask.any():
            centers.append((low + high) / 2.0)
            medians.append(float(np.median(error[mask])))
    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    ax.scatter(true, error, s=9, alpha=0.28, linewidths=0, color=ACCENT_BLUE, rasterized=True)
    ax.axhline(0, color="black", linestyle="--", linewidth=1.1)
    ax.plot(centers, medians, color=MEDIAN_COLOR, linewidth=1.8, marker="o",
            label="Median residual")
    ax.set_xlim(0, cfg["eval_max"])
    ax.set_ylim(*error_ylim(error))
    ax.set_xlabel("RH95 from GEDI waveforms (m)")
    ax.legend(framealpha=0.70)
    export_figure(fig, output_dir, "05_residuals_vs_height")
    plt.show()
    plt.close(fig)


def plot_absolute_error_cdf(frame, forest, output_dir):
    absolute = np.sort(np.abs(
        frame["prediction"].to_numpy(float) - frame["rh95"].to_numpy(float)
    ))
    cumulative = np.arange(1, len(absolute) + 1) / len(absolute)
    fig, ax = plt.subplots(figsize=(7.8, 4.6))
    ax.plot(absolute, cumulative, color=ACCENT_ORANGE, linewidth=2.1)
    for threshold in (1, 2, 3, 5, 10):
        fraction = 100.0 * float(np.mean(absolute <= threshold))
        ax.axvline(threshold, color="0.4", linestyle="--", linewidth=0.8, alpha=0.55)
        ax.text(threshold, 0.035, f"{fraction:.0f}% ≤ {threshold} m",
                rotation=90, va="bottom", ha="right", fontsize=8)
    ax.set_xlim(0, max(float(np.percentile(absolute, 99.5)), 5.0))
    ax.set_ylim(0, 1.01)
    ax.set_xlabel("|Prediction − GEDI| (m)")
    export_figure(fig, output_dir, "06_absolute_error_cdf")
    plt.show()
    plt.close(fig)


def run_forest_report(forest: str) -> dict:
    frame = load_phase2_test(forest)
    cfg = SITES[forest]
    output_dir = OUTPUT_ROOT / cfg["ecosystem"].replace(" ", "_") / forest
    metric = compute_metrics(frame)
    print(f"{forest} | Phase 2 TEST | n={len(frame):,} | output={output_dir}")
    plot_scatter(frame, forest, output_dir)
    plot_height_distribution_error(frame, forest, output_dir)
    plot_height_error_bins(frame, forest, output_dir)
    plot_signed_error(frame, forest, output_dir)
    plot_residuals(frame, forest, output_dir)
    plot_absolute_error_cdf(frame, forest, output_dir)
    pd.DataFrame([metric]).assign(
        forest=forest,
        ecosystem=cfg["ecosystem"],
        eval_min_m=cfg["eval_min"],
        eval_max_m=cfg["eval_max"],
        prediction_column="pred_on_growthloss",
    ).to_csv(output_dir / "metrics.csv", index=False)
    return {"forest": forest, "ecosystem": cfg["ecosystem"], **metric}


In [ ]:
factorial_metrics = pd.DataFrame([run_forest_report(name) for name in VARIANTS])
factorial_metrics["phase1_delta"] = 1.0
factorial_metrics["phase1_parent"] = factorial_metrics["forest"].map(lambda n: VARIANTS[n]["parent_rule"])
factorial_metrics["phase2_delta"] = factorial_metrics["forest"].map(lambda n: VARIANTS[n]["p2_delta"])
factorial_metrics.to_csv(EXPERIMENT / "02_factorial_test_metrics.csv", index=False)
display(factorial_metrics[["forest", "phase1_parent", "phase2_delta", "n", "mae", "rmse", "r2", "bias", "slope", "corr", "std_ratio"]])


## 7. Factor contrasts and decision table

The contrasts below isolate the Phase 2 delta effect within each fixed parent,
and the parent effect within each fixed Phase 2 delta. Lower MAE/RMSE and
smaller absolute deviations of slope and SD ratio from one are favourable.


In [ ]:
# Descriptive contrasts only; no TEST-guided checkpoint selection is performed.
factorial_metrics = pd.read_csv(EXPERIMENT / '02_factorial_test_metrics.csv')
m = factorial_metrics.set_index('forest')
rows = []
for parent in ('best_any', 'best_slope'):
    d1, d3 = f'{parent}__p2_delta1', f'{parent}__p2_delta3'
    rows.append({'contrast': f'Phase 2 delta 1 minus 3 | parent={parent}',
                 **{f'delta_{k}': m.loc[d1, k] - m.loc[d3, k] for k in ('mae', 'rmse', 'r2', 'bias', 'slope', 'std_ratio')}})
for delta in (1, 3):
    slope, any_ = f'best_slope__p2_delta{delta}', f'best_any__p2_delta{delta}'
    rows.append({'contrast': f'best_slope minus best_any | Phase 2 delta={delta}',
                 **{f'delta_{k}': m.loc[slope, k] - m.loc[any_, k] for k in ('mae', 'rmse', 'r2', 'bias', 'slope', 'std_ratio')}})
contrasts = pd.DataFrame(rows)
contrasts.to_csv(EXPERIMENT / '03_factorial_contrasts.csv', index=False)
display(contrasts)
decision = factorial_metrics.copy()
decision['abs_slope_from_1'] = (decision['slope'] - 1).abs()
decision['abs_std_ratio_from_1'] = (decision['std_ratio'] - 1).abs()
decision.to_csv(EXPERIMENT / '04_descriptive_metrics_no_model_selection.csv', index=False)
display(decision[['forest', 'mae', 'rmse', 'r2', 'slope', 'std_ratio', 'abs_slope_from_1', 'abs_std_ratio_from_1']])
print('Audit complete. Do not use TEST results to launch or select additional variants.')


## 8. Interpretation boundary

This completed factorial demonstrates a parent-checkpoint × Phase 2 Huber-delta interaction on a common 1,799-shot TEST support. It is a descriptive audit of the already frozen workflow, not a new TEST-based selection step. The designated production lineage must therefore remain unchanged unless a new choice is made prospectively on VAL and evaluated on a new untouched test support.
